In [ ]:
import os

from hera.shared import global_config
from hera.workflows import (
    DAG,
    Artifact,
    Parameter,
    Script,
    Volume,
    Workflow,
    script,  # pyright: ignore[reportUnknownVariableType]
)
from hera.workflows import models as m
from hera.workflows.archive import NoneArchiveStrategy

global_config.set_class_defaults(  # pyright: ignore
    Script, image="ghcr.io/diamondlightsource/httomo:latest"
)

In [ ]:
import json


@script(pod_spec_patch=json.dumps({"containers":
                    [{"name":"main",
                    "resources":
                        {"limits":{"cpu":"{{inputs.parameters.nprocs}}",
                                    "memory":"{{inputs.parameters.memory}}",
                                    "nvidia.com/gpu":"{{inputs.parameters.nprocs}}"},
                        "requests":{"cpu":"{{inputs.parameters.nprocs}}",
                                    "memory":"{{inputs.parameters.memory}}",
                                    "nvidia.com/gpu":"{{inputs.parameters.nprocs}}"}}}]}),
            tolerations=[
                m.Toleration(key="nvidia.com/gpu",operator="Exists",effect="NoSchedule"),
                m.Toleration(key="nodetype",operator="Equal",value="gpu",effect="NoSchedule"),
                m.Toleration(key="nodegroup",operator="Equal",value="workflows",effect="NoSchedule")],
    command=["/opt/conda/bin/python"],
    volume_mounts=[
        m.VolumeMount(name="session", mount_path="{{workflow.parameters.visitdir}}"),
        m.VolumeMount(name="tmpdir", mount_path="/tmp"),
    ],
    outputs=[
        Parameter(
            name="out-path",
            value_from=m.ValueFrom(
                path="/tmp/parameters.json"
            )
        )
    ]
)
def tomo_recon(
    config: str, input: str, output: str, recon_outdir_name: str, nprocs: int, memory: str
):
    import json
    import subprocess


    subprocess.check_call([
        "/opt/conda/bin/mpirun",
        "-n",
        str(nprocs),
        "/opt/conda/bin/python",
        "-m",
        "httomo",
        "run",
        "--pipeline-format",
        "json",
        "--output-folder-name",
        recon_outdir_name,
        input,
        str(config),
        output
    ])

    with open("/tmp/parameters.json", "w") as f:
        json.dump(f"{output}/{recon_outdir_name}", f)

In [ ]:
@script(
    command=["/opt/conda/bin/python"],
    volume_mounts=[
        m.VolumeMount(name="session", mount_path="{{workflow.parameters.visitdir}}"),
        m.VolumeMount(name="tmpdir", mount_path="/tmp"),
    ],
    outputs=[
        Artifact(
            name="recon",
            path="{{inputs.parameters.tmpdir_path}}/{{inputs.parameters.metadata_filename}}",
            archive=NoneArchiveStrategy()
        )
    ]
)
def convert_recon_data_format(recon_dir_path: str,
                              tmpdir_path: str = "/tmp",
                              raw_recon_filename: str = "recon.raw",
                              metadata_filename: str = "metadata.json"):
    import json
    from pathlib import Path

    import h5py



    RAW_RECON_PATH = f"{tmpdir_path}/{raw_recon_filename}"
    HDF5_RECON_DIR = Path(recon_dir_path)
    HDF5_RECON_FILENAME_PATTERN = "*-httomolib-rescale_to_int.h5"
    hdf5_recon_data_path = list(HDF5_RECON_DIR.glob(HDF5_RECON_FILENAME_PATTERN))[0]

    with h5py.File(hdf5_recon_data_path, "r") as f:
        data = f["/data"][:]
        data.tofile(RAW_RECON_PATH)

    METADATA_PATH = f"{tmpdir_path}/{metadata_filename}"

    order = "C" if data.flags.c_contiguous else "F"
    metadata = {"shape": list(data.shape), "dtype": str(data.dtype), "order": order}
    with open(METADATA_PATH, "w") as f:
        f.write(json.dumps(metadata, indent=2))

In [ ]:
from hera.workflows.volume import HostPathVolume

with Workflow(
    name="visr-recon-with-python-interface-to-workflows",
    entrypoint="workflowentry",
    api_version="argoproj.io/v1alpha1",
    kind="WorkflowTemplate",
    labels={"workflows.diamond.ac.uk/science-group-imaging": "true"},
    annotations={
        "workflows.argoproj.io/title": "ViSR recon",
        "workflows.argoproj.io/description": """ViSR recon
example.yaml""",
        "workflows.diamond.ac.uk/repository": "https://github.com/DiamondLightSource/imaging-python-workflow-test",
    },
    volumes=[
        Volume(name="tmpdir", mount_path="/tmp/", size="1Gi"),
        HostPathVolume(name="session",
                       path="{{workflow.parameters.visitdir}}",
                       type="Directory"),
    ],
    arguments=m.Arguments(
        parameters=[
            m.Parameter(
                name="visitdir",
                value_from=m.ValueFrom(
                    config_map_key_ref=m.ConfigMapKeySelector(
                        name="sessionspaces",
                        key="data_directory"
                    )
                )
            )
        ]
    )
) as w:
    with DAG(name="workflowentry"):
        config = """[
  {
    "method": "standard_tomo",
    "module_path": "httomo.data.hdf.loaders",
    "parameters": {
      "data_path": "/entry1/tomo_entry/data/data",
      "image_key_path": "/entry1/tomo_entry/instrument/detector/image_key",
      "rotation_angles": {
        "data_path": "/entry1/tomo_entry/data/rotation_angle"
      },
      "preview": {
        "detector_y": {
          "start": 100,
          "stop": 102
        }
      }
    }
  },
  {
    "method": "remove_outlier",
    "module_path": "tomopy.misc.corr",
    "parameters": {
      "dif": 0.1,
      "size": 3,
      "axis": "auto"
    }
  },
  {
    "method": "find_center_vo",
    "module_path": "httomolibgpu.recon.rotation",
    "parameters": {
      "ind": null,
      "smin": -50,
      "smax": 50,
      "srad": 6,
      "step": 0.25,
      "ratio": 0.5,
      "drop": 20
    },
    "id": "centering",
    "side_outputs": {
      "cor": "centre_of_rotation"
    }
  },
  {
    "method": "normalize",
    "module_path": "httomolibgpu.prep.normalize",
    "parameters": {
      "cutoff": 10.0,
      "minus_log": true,
      "nonnegativity": false,
      "remove_nans": false
    }
  },
  {
    "method": "FBP3d_tomobar",
    "module_path": "httomolibgpu.recon.algorithm",
    "parameters": {
      "center": "${{centering.side_outputs.centre_of_rotation}}",
      "filter_freq_cutoff": 0.6,
      "recon_size": null,
      "recon_mask_radius": null
    },
    "save_result": true
  }
]"""
        recon = tomo_recon(
            arguments={
                "config": config,
                "input": "/dls/i12/data/2025/cm40628-3/rawdata/188700.nxs",
                "output": "/dls/i12/data/2025/cm40628-3/processing/yousef/workflows",
                "recon_outdir_name": "sweep-run", "nprocs": 1, "memory": "1Gi"
            }
        )
        convert = convert_recon_data_format(
            arguments={
                "recon_dir_path": recon.get_parameter("out-path"),
            }
        )
        recon >> convert  # pyright: ignore




In [ ]:

with open("src/python_interface_to_workflows/templates/realexample.txt", "w") as div:
    div.write(w.to_yaml())  # pyright: ignore[reportUnknownMemberType]

In [ ]:
from python_interface_to_workflows.submit_workflow import submit_workflow

await submit_workflow(w)